# Music Genre Classification Data Transformation & Preprocessing

## Objective

This notebook covers the **Data Transformation** phase, preparing everything for our PyTorch model.

Our pipeline follows three stages:

1. **EDA** — completed (class distribution, waveforms, spectrograms, MFCCs, 
   corrupted/duplicate file detection)
2. **SQL schema & cleaning** — completed (metadata table built from EDA findings: 
   duplicate/corrupted flags, train/val/test split assigned at track level)
3. **Data Transformation** — this notebook (preprocessing pipeline: trim/pad, 
   windowing, MFCC extraction, normalization, augmentation)

## Goal
Create a PyTorch Dataset/DataLoader from clean SQL metadata we expect by the end of this notebook, we will have a working PyTorch `Dataset` and `DataLoader`
### Workflow — preprocess_pytorch.ipynb

- **Set random seed (42)** — numpy, torch, and Python's `random` module, for reproducible splits and results
- **Connect to SQL** — query `vw_clean_tracks` to pull `file_path`, `label`, `genre_id`, `split` for all 971 clean tracks
- **Trim/pad audio** — standardize every clip to exactly 30 seconds, save to `processed/`
- **GTZANDataset class** — for each track requested:
  - slice a 3-second window (random for train, fixed/centered for val/test)
  - apply augmentation — time shift, noise, frequency masking (train only)
  - compute the full 2D MFCC array (n_mfcc=20)
  - normalize using train-only mean/std
  - add channel dimension for Conv2d input
- **Compute normalization stats** — mean/std from train split only, reused across all splits
- **Recreate train/val/test datasets** — with normalization applied
- **Wrap in DataLoader** — verify batch shape and value range before moving to model development

In [1]:
# --- Standard library ---
import os
import random
from pathlib import Path

# --- Data & numeric ---
import numpy as np
import pandas as pd

# --- Audio processing ---
import librosa
import soundfile as sf

# --- Database connection ---
import psycopg2
from dotenv import load_dotenv

# --- PyTorch ---
import torch
from torch.utils.data import Dataset, DataLoader

# --- SQLAlchemy ---
from sqlalchemy import create_engine

In [2]:
## Set random seeds for reproducibility 
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Conexión SQL (971 clean_tracks)

In [3]:

# load environment variables from .env file 
load_dotenv()

# read DB connection details from environment variables
DB_NAME = os.getenv("DB_NAME", "music_genre_db")  # database name, default to music_genre_db
DB_USER = os.getenv("DB_USER", "ingxrodriguez")    # postgres user
DB_PASSWORD = os.getenv("DB_PASSWORD")              # postgres password, must come from .env, never hardcoded
DB_HOST = os.getenv("DB_HOST", "localhost")         # db host
DB_PORT = os.getenv("DB_PORT", "5432")              # db port

# create a SQLAlchemy engine using those credentials
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# query the clean tracks view: excludes corrupted_flagged and duplicate_flagged rows
query = "SELECT track_id, file_path, label, genre_id, split FROM vw_clean_tracks;"

# read the query result directly into a pandas DataFrame, using the engine 
clean_tracks_df = pd.read_sql(query, engine)

# quick sanity check: confirm row count and preview the first few rows
print("Total clean tracks:", len(clean_tracks_df))
clean_tracks_df.head()

Total clean tracks: 971


,track_id,file_path,label,genre_id,split
0,16,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train
1,41,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train
2,3,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train
3,78,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,val
4,28,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,val


# Trim/pad a 30 seg

### Why we trim/pad to 30 seconds

Our EDA showed that GTZAN clips are almost all ~30 seconds long, but not 
perfectly uniform a few are slightly shorter or longer. Since our model 
extracts MFCCs as full 2D arrays (coefficients × time), every clip must 
have the exact same number of audio samples; otherwise the resulting MFCC 
arrays will have different time dimensions and can't be batched together 
by the DataLoader.

To fix this, we standardize every clip to exactly 30 seconds (22050 Hz × 30 
= 661,500 samples):
- Clips longer than 30s are **trimmed** down to the target length.
- Clips shorter than 30s are **padded** with silence (zeros) at the end.

Original files are never modified  trimmed/padded copies are saved to a 
separate `processed/` folder, preserving the raw dataset as our permanent 
reference.

In [4]:
# native sample rate confirmed identical across all readable files during EDA (22050 Hz)
TARGET_SR = 22050

# standard clip length decided during cleaning: 30 seconds (matches original GTZAN clip length)
TARGET_DURATION_SEC = 30

# convert target duration to number of samples (30 sec * 22050 samples/sec)
TARGET_LENGTH_SAMPLES = TARGET_SR * TARGET_DURATION_SEC

# root folder where original, untouched audio files live
DATA_ROOT = Path("../Data_Music")

# new folder for trimmed/padded copies -- originals are never overwritten
PROCESSED_DIR = DATA_ROOT / "processed"


def trim_or_pad(y, target_length):
    """Trim audio array to target_length, or pad with zeros (silence) if shorter."""
    current_length = len(y)  # number of samples in the loaded audio

    if current_length > target_length:
        # clip is longer than target: cut it down to exactly target_length samples
        return y[:target_length]
    elif current_length < target_length:
        # clip is shorter than target: pad the end with zeros (silence) up to target_length
        pad_amount = target_length - current_length
        return np.pad(y, (0, pad_amount), mode="constant")
    else:
        # already exactly the target length, no change needed
        return y


# will store the new, trimmed file path for each track so we can save it back to the dataframe
processed_paths = []

# loop through every clean track from our SQL query
for row in clean_tracks_df.itertuples(index=False):
    # load the original audio at its verified native sample rate (sr=None reads actual rate, doesn't impose one)
    y, sr = librosa.load(row.file_path, sr=None)

    # apply trim/pad so every clip has exactly TARGET_LENGTH_SAMPLES samples
    y_fixed = trim_or_pad(y, TARGET_LENGTH_SAMPLES)

    # build the output subfolder path, mirroring genre structure: processed/<label>/
    genre_subdir = PROCESSED_DIR / row.label
    genre_subdir.mkdir(parents=True, exist_ok=True)  # create folder if it doesn't exist yet

    # build the output file path using the same filename as the original
    out_path = genre_subdir / Path(row.file_path).name

    # write the trimmed/padded audio to disk (original file is untouched)
    sf.write(out_path, y_fixed, sr)

    # record the new path so we can update our dataframe afterward
    processed_paths.append(str(out_path))

# add the processed file path as a new column, keeping the original file_path column intact
clean_tracks_df["processed_path"] = processed_paths

# confirm how many files were processed
print(f"Trimmed/padded {len(clean_tracks_df)} files to {TARGET_DURATION_SEC} sec each.")
clean_tracks_df.head()

Trimmed/padded 971 files to 30 sec each.


,track_id,file_path,label,genre_id,split,processed_path
0,16,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train,../Data_Music/processed/blues/blues.00015.wav
1,41,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train,../Data_Music/processed/blues/blues.00040.wav
2,3,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train,../Data_Music/processed/blues/blues.00002.wav
3,78,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,val,../Data_Music/processed/blues/blues.00077.wav
4,28,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,val,../Data_Music/processed/blues/blues.00027.wav


### What does our class GTZANDataset

We created a class to wrap our entire feature extraction pipeline into a single reusable PyTorch Dataset.

1. **Filter by split** (`__init__`): keep only the rows of `clean_tracks_df` 
   belonging to this split ("train", "val", or "test").

2. **Load the audio** (`__getitem__`, step 1): load the already 
   trimmed/padded 30-second file from `processed_path`, using its verified 
   native sample rate (22050 Hz).

3. **Slice a 3-second window** (`_get_window`): cut a 3-second slice out of 
   the 30-second clip, *before* computing anything else.
   - **Train**: pick a random start point every time — so the same song can 
     contribute a different 3-second slice on different epochs, giving the 
     model more effective variety from the same 677 tracks.
   - **Val/test**: always use the same centered window — evaluation must be 
     reproducible, so we don't want a random target that changes every run.

4. **Augment the raw audio — train only** (`_time_shift`, `_add_noise`): 
   shift the window slightly in time and add a touch of background noise. 
   Only applied when `split == "train"` — val/test must reflect real, 
   unaltered audio.

5. **Compute the MFCC** (`librosa.feature.mfcc`): extract the full 2D MFCC 
   array over just the 3-second window — shape `(20, T)`, coefficients × 
   time frames, **not averaged**. This preserves the temporal structure a 
   CNN needs, and is ~10x cheaper to compute than over the full 30 seconds, 
   since we sliced *before* this step, not after.

6. **Frequency masking — train only** (`_frequency_mask`): zero out a random 
   band of MFCC coefficient rows, forcing the model to not over-rely on any 
   single frequency range. Applied directly to the MFCC array, only for train.

7. **Normalize**: apply `(mfcc - mean) / std`, using `MFCC_MEAN`/`MFCC_STD` 
   computed from the train split only — the same two numbers are reused for 
   val and test, to avoid leaking information about the evaluation data into 
   training.

8. **Add the channel dimension** (`unsqueeze(0)`): reshape from `(20, T)` to 
   `(1, 20, T)` — the format a `Conv2d` layer expects (1 channel, like a 
   grayscale image, since MFCC values aren't RGB color channels).

9. **Return** the final MFCC tensor and its numeric `genre_id` label — ready 
   to be batched by the `DataLoader` and fed directly into the CNN.



In [5]:
class GTZANDataset(Dataset):
    #PyTorch Dataset that loads a 3-second window of audio and returns its normalized MFCC array + genre label.
    #Train uses a random window each call (dynamic variety); val/test use a fixed centered window (reproducible eval).
    #Applies data augmentation (time shift, noise, frequency masking) ONLY when split == 'train'.

    def __init__(self, metadata_df, split, n_mfcc=20, mfcc_mean=None, mfcc_std=None,
                 window_duration_sec=3, sample_rate=22050):
        # keep only the rows belonging to this split ("train", "val", or "test")
        self.df = metadata_df[metadata_df["split"] == split].reset_index(drop=True)

        # store the split name used below to decide augmentation AND windowing strategy (train only gets randomness)
        self.split = split

        # number of MFCC coefficients to extract per frame (standardized to 20, matches EDA)
        self.n_mfcc = n_mfcc

        # normalization stats  MUST be computed from the train split only, then reused for all splits
        self.mfcc_mean = mfcc_mean
        self.mfcc_std = mfcc_std

        # window length in samples (3 sec * 22050 Hz = 66150 samples) -- much cheaper to run MFCC on than 30 sec
        self.window_length_samples = int(window_duration_sec * sample_rate)

    def __len__(self):
        # total number of tracks available in this split
        return len(self.df)

    def _get_window(self, y):
        """Slice a 3-second window out of the raw (already 30-sec trimmed/padded) audio,
        BEFORE computing the MFCC -- this is what makes the MFCC extraction ~10x cheaper,
        since librosa's FFT-based MFCC runs on a much shorter signal."""
        max_start = len(y) - self.window_length_samples

        if self.split == "train":
            # TRAIN: pick a different random start point every time this track is requested,
            # so the same song can contribute a different 3-sec slice across different epochs
            start = random.randint(0, max_start)
        else:
            # VAL/TEST: always use the same centered window, so evaluation is reproducible
            # across runs we want a fixed target to measure the model against, not a moving one
            start = max_start // 2

        return y[start:start + self.window_length_samples]

    def _time_shift(self, y):
        """Shift the raw audio left/right by a random amount, wrapping around the ends."""
        # pick a random shift amount: up to 10% of the clip length, in either direction
        max_shift = int(0.1 * len(y))
        shift_amount = random.randint(-max_shift, max_shift)
        # np.roll wraps values around the edges instead of losing them
        return np.roll(y, shift_amount)

    def _add_noise(self, y, noise_level=0.005):
        """Add a small amount of random Gaussian noise to the raw audio."""
        # generate noise the same shape as y, scaled to be subtle relative to the audio
        noise = np.random.randn(len(y)) * noise_level
        return y + noise

    def _frequency_mask(self, mfcc, max_mask_width=4):
        """Zero out a random contiguous band of MFCC coefficient rows (frequency masking)."""
        # mfcc shape is (n_mfcc, T) -- we mask rows (frequency axis), not time
        n_mfcc, T = mfcc.shape
        # pick a random width for the masked band, up to max_mask_width rows
        mask_width = random.randint(1, max_mask_width)
        # pick a random starting row so the whole band fits inside n_mfcc
        start = random.randint(0, n_mfcc - mask_width)
        # copy so we don't modify the original array in place
        mfcc_masked = mfcc.copy()
        mfcc_masked[start:start + mask_width, :] = 0.0
        return mfcc_masked

    def __getitem__(self, idx):
        # look up the row for this index in our filtered split dataframe
        row = self.df.iloc[idx]

        # load the already trimmed/padded 30-sec audio file (sr=None preserves the verified native rate)
        y, sr = librosa.load(row["processed_path"], sr=None)

        # slice down to a 3-second window BEFORE computing MFCC (this is the compute-saving step)
        y = self._get_window(y)

        # augmentation on the raw 3-sec window -- ONLY applied to the train split
        if self.split == "train":
            y = self._time_shift(y)   # randomly shift the window in time
            y = self._add_noise(y)    # add a small amount of background noise

        # compute the full 2D MFCC array over the 3-sec window: shape = (n_mfcc, T)  NOT averaged over time
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=self.n_mfcc)

        # frequency masking works directly on the MFCC array -- ONLY applied to the train split
        if self.split == "train":
            mfcc = self._frequency_mask(mfcc)

        # convert the MFCC numpy array into a PyTorch tensor (float32, standard for model input)
        mfcc_tensor = torch.tensor(mfcc, dtype=torch.float32)

        # apply normalization using the train-only mean/std, if provided
        if self.mfcc_mean is not None and self.mfcc_std is not None:
            mfcc_tensor = (mfcc_tensor - self.mfcc_mean) / self.mfcc_std

        # add the channel dimension a Conv2d layer expects: (n_mfcc, T) -> (1, n_mfcc, T)
        mfcc_tensor = mfcc_tensor.unsqueeze(0)

        # genre_id is our numeric label (matches the genre_id column from music_genre table)
        label = torch.tensor(row["genre_id"], dtype=torch.long)

        # return the normalized MFCC tensor and its label -- this is what the DataLoader will batch together
        return mfcc_tensor, label

### Why we use all three augmentation techniques together

Each technique targets a different kind of variation the model might 
encounter in real world audio, and they're cheap to compute, so combining 
them gives more robustness without much added cost:

- **Time shifting** makes the model invariant to *when* a pattern occurs 
  in the clip a riff or beat shouldn't be recognized only if it starts 
  at the exact same millisecond.
- **Adding noise** simulates imperfect recording conditions (background 
  noise, compression artifacts) so the model doesn't rely on unrealistically 
  clean audio.
- **Frequency masking** prevents the model from over relying on any single 
  frequency band, forcing it to learn more general, distributed patterns 
  across the MFCC.

Together, these reduce overfitting on our relatively small training set 
(677 tracks) by exposing the model to more variation than the raw dataset 
alone provides. All three are applied **only to the training split** 
val and test must reflect real, unaltered data, since their purpose is to 
measure how well the model generalizes to genuinely unseen audio.

### Compute normalization stats (train split only)

Creates a temporary `GTZANDataset` for the train split with no `mfcc_mean`/`mfcc_std` 
passed in, so it returns raw, unnormalized MFCCs. Loops through every training 
track, collects its raw MFCC tensor, and stacks them all into one tensor to compute 
a single global mean and standard deviation.

These two numbers (`MFCC_MEAN`, `MFCC_STD`) are computed **only** from train — never 
from val or test and will be reused to normalize all three splits. This prevents 
data leakage: no information about the validation or test sets influences how the 
data gets normalized.

In [6]:
# calculate the mean and std of the MFCCs from the train split only, to be used for normalization in all splits
# create a temporary train-only dataset with no mean/std passed in, so MFCCs come back raw (not normalized yet)
train_dataset_raw = GTZANDataset(clean_tracks_df, split="train")

# empty list to collect every raw MFCC tensor from the train split
all_train_mfccs = []
# loop through every track in the train split
for i in range(len(train_dataset_raw)):
# get the raw MFCC tensor for this track, ignore the label (we only need MFCC values here)
    mfcc_tensor, _ = train_dataset_raw[i]
# add this track's MFCC tensor to our list
    all_train_mfccs.append(mfcc_tensor)
# stack the list of individual MFCC tensors into one big tensor: shape (num_train_tracks, 20, T)
stacked_train_mfccs = torch.stack(all_train_mfccs)
# compute a single global mean across all train MFCC values (all tracks, all coefficients, all time frames)
MFCC_MEAN = stacked_train_mfccs.mean()
# compute a single global standard deviation across all train MFCC values
MFCC_STD = stacked_train_mfccs.std()
# print the computed stats so we can document and reproduce them later
print("Train MFCC mean:", MFCC_MEAN.item())
print("Train MFCC std:", MFCC_STD.item())

Train MFCC mean: -0.6697229146957397
Train MFCC std: 38.162445068359375


### Recreate datasets with normalization applied

Creates the three final `GTZANDataset` objects — one per split this time passing 
in `MFCC_MEAN` and `MFCC_STD` (computed from train only), so every MFCC returned by 
`__getitem__` is now normalized.

Then runs a quick sanity check: pulls the first training sample to confirm its 
shape/label look right, prints the size of each split (expected: 677 / 141 / 153), 
and prints the normalized sample's mean/std — these should land close to 0 and 1 
respectively, confirming the normalization was applied correctly.

In [7]:
# calling our class GTZANDataset to create 3 objects/instances for train, val, and test splits, 
# passing in the mean and std computed from train only, so all 3 splits normalize the same way
train_dataset = GTZANDataset(clean_tracks_df, split="train", mfcc_mean=MFCC_MEAN, mfcc_std=MFCC_STD)
val_dataset = GTZANDataset(clean_tracks_df, split="val", mfcc_mean=MFCC_MEAN, mfcc_std=MFCC_STD)
test_dataset = GTZANDataset(clean_tracks_df, split="test", mfcc_mean=MFCC_MEAN, mfcc_std=MFCC_STD)

# test that the datasets work correctly by retrieving the first sample from train and checking its shape/label
sample_mfcc, sample_label = train_dataset[0]
# confirm how many tracks landed in each split, and confirm normalization worked (mean near 0, std near 1)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))
print("Normalized sample MFCC mean:", sample_mfcc.mean().item())
print("Normalized sample MFCC std:", sample_mfcc.std().item())

Train size: 677
Val size: 141
Test size: 153
Normalized sample MFCC mean: -0.048923056572675705
Normalized sample MFCC std: 1.1722360849380493


### Wrap datasets in a DataLoader and verify a batch

Wraps each `GTZANDataset` in a `DataLoader`. Train shuffles tracks every epoch 
(`shuffle=True`) so the model sees genres in a different order each time; val/test 
don't need shuffling since they're only used for evaluation, not training.

Pulls one batch from `train_loader` as a final end-to-end check before moving to 
model development — confirming:
- **Batch MFCC shape**: `(32, 1, 20, T)` — batch size, channel, MFCC coefficients, 
  time frames (T depends on the 3-second window)
- **Batch labels shape**: `(32,)` — one genre label per track
- **Value range**: should be centered near 0, consistent with our normalization

In [8]:
# batch size: how many tracks are grouped together per training step
BATCH_SIZE = 32

# train loader: shuffle=True so the model sees genres in a different random order each epoch
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# val/test loaders: shuffle=False -- no need to shuffle since we're only evaluating, not training
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# pull one batch from the train loader to verify everything works end-to-end
batch_mfcc, batch_labels = next(iter(train_loader))

# expected shape: (BATCH_SIZE, n_mfcc, T) -- e.g. (32, 20, 1292)
print("Batch MFCC shape:", batch_mfcc.shape)

# expected shape: (BATCH_SIZE,) -- one genre_id per track in the batch
print("Batch labels shape:", batch_labels.shape)

# sanity check on value range -- should be centered near 0 given our normalization
print("Batch MFCC value range: min =", batch_mfcc.min().item(), " max =", batch_mfcc.max().item())

Batch MFCC shape: torch.Size([32, 1, 20, 130])
Batch labels shape: torch.Size([32])
Batch MFCC value range: min = -7.876141548156738  max = 4.073089122772217


# Conclusion
The preprocessing pipeline is fully verified end-to-end: metadata comes from a clean, deduplicated, leakage-free SQL split (677/141/153 tracks); each batch returns a (32, 1, 20, 130) tensor 32 tracks, 1 channel, 20 MFCC coefficients, 130 time frames from a 3-second window with values centered near 0 and a std near 1, confirming normalization worked correctly. Data augmentation (time shift, noise, frequency masking) is applied only to the training split, and the train/val/test split is assigned at the original-track level in SQL, before any windowing occurs so no audio from a given song can appear in more than one split. The pipeline is ready to feed directly into a CNN for Sprint 3 model development.